In [1]:
from google.colab import userdata, drive
import os, json, base64, mimetypes, requests
from pathlib import Path
from PIL import Image
from PIL.ExifTags import TAGS, GPSTAGS

# ── Mount Google Drive ────────────────────────────────────────────────────────
drive.mount('/content/drive', force_remount=True)

# ── Config ────────────────────────────────────────────────────────────────────
OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY').strip()
MODEL       = "google/gemini-3.1-flash-lite"
DRIVE_BASE  = Path("/content/drive/MyDrive/team")
PHOTOS_DIR  = DRIVE_BASE / "photos"
OUTPUT_JSON = DRIVE_BASE / "descriptions.json"
IMAGE_EXTS  = {".jpg", ".jpeg", ".png", ".webp", ".gif", ".bmp"}
HEADERS     = {
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
    "Content-Type":  "application/json",
    "HTTP-Referer":  "https://colab.research.google.com",
    "X-Title":       "Image Describer",
}

print(f"📁  Drive base : {DRIVE_BASE}")
print(f"🖼️   Photos dir : {PHOTOS_DIR}")
print(f"📄  Output JSON: {OUTPUT_JSON}")

# ── EXIF Helpers ──────────────────────────────────────────────────────────────
def get_gps(gps_info: dict) -> dict | None:
    """Convert raw GPS EXIF data to lat/lon decimal degrees."""
    def to_decimal(values, ref):
        d, m, s = values
        decimal = float(d) + float(m) / 60 + float(s) / 3600
        if ref in ("S", "W"):
            decimal = -decimal
        return round(decimal, 6)
    try:
        decoded = {GPSTAGS.get(k, k): v for k, v in gps_info.items()}
        lat = to_decimal(decoded["GPSLatitude"], decoded["GPSLatitudeRef"])
        lon = to_decimal(decoded["GPSLongitude"], decoded["GPSLongitudeRef"])
        result = {"latitude": lat, "longitude": lon}
        if "GPSAltitude" in decoded:
            result["altitude_m"] = round(float(decoded["GPSAltitude"]), 1)
        return result
    except Exception:
        return None

def extract_metadata(path: Path) -> dict:
    """Extract useful EXIF metadata from an image. Returns {} if none found."""
    meta = {}
    try:
        img = Image.open(path)
        meta["width"]  = img.width
        meta["height"] = img.height
        meta["format"] = img.format or path.suffix.lstrip(".").upper()

        exif_raw = img._getexif()
        if not exif_raw:
            return meta

        exif = {TAGS.get(k, k): v for k, v in exif_raw.items()}

        # Camera / capture info
        for field, key in [
            ("Make",             "camera_make"),
            ("Model",            "camera_model"),
            ("LensModel",        "lens_model"),
            ("Software",         "software"),
            ("DateTime",         "datetime_original"),
            ("DateTimeOriginal", "datetime_original"),
            ("ExposureTime",     "exposure_time"),
            ("FNumber",          "f_number"),
            ("ISOSpeedRatings",  "iso"),
            ("FocalLength",      "focal_length_mm"),
            ("Flash",            "flash"),
            ("WhiteBalance",     "white_balance"),
            ("ExposureProgram",  "exposure_program"),
            ("MeteringMode",     "metering_mode"),
            ("SceneCaptureType", "scene_type"),
        ]:
            if field in exif and exif[field] is not None:
                val = exif[field]
                # Convert IFDRational to float
                try:
                    val = float(val)
                    if field == "ExposureTime" and val < 1:
                        val = f"1/{round(1/val)}"
                    elif field in ("FNumber", "FocalLength"):
                        val = round(val, 1)
                except Exception:
                    val = str(val)
                meta[key] = val

        # GPS
        if "GPSInfo" in exif:
            gps = get_gps(exif["GPSInfo"])
            if gps:
                meta["gps"] = gps

    except Exception as e:
        meta["metadata_error"] = str(e)

    return meta

# ── Core Helpers ──────────────────────────────────────────────────────────────
def load_results() -> dict:
    if OUTPUT_JSON.exists():
        with open(OUTPUT_JSON, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}

def save_results(results: dict):
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"💾  Saved → {OUTPUT_JSON}")

def image_to_base64(path: Path) -> tuple[str, str]:
    mime = mimetypes.guess_type(path)[0] or "image/jpeg"
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode(), mime

def call_openrouter(messages: list) -> str:
    resp = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers=HEADERS,
        json={
            "model": MODEL,
            "max_tokens": 1024,
            "messages": messages,
            "system": (
                "You are a precise image captioning assistant producing training data for a vision LLM. "
                "Always write in plain, continuous prose paragraphs. "
                "Never use markdown, headers, bullet points, bold, italics, or any formatting symbols. "
                "Never add preambles, commentary, or meta-text. Output only the requested content."
            ),
        },
        timeout=60,
    )
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]["content"].strip()

# ── Main ──────────────────────────────────────────────────────────────────────
def process_images():
    if not PHOTOS_DIR.exists():
        print(f"\n❌  Photos folder not found: {PHOTOS_DIR}")
        print("\n📂  Available in Drive base:")
        for item in DRIVE_BASE.iterdir():
            print(f"    {item.name}")
        return

    images = sorted(p for p in PHOTOS_DIR.iterdir() if p.suffix.lower() in IMAGE_EXTS)
    if not images:
        print(f"⚠️  No images found in {PHOTOS_DIR}")
        return

    results = load_results()
    done_count = sum(1 for v in results.values() if v.get("status") == "done")
    skipped, processed, failed = 0, 0, 0

    print(f"\n📂  Found {len(images)} image(s) in '{PHOTOS_DIR.name}'")
    print(f"📋  Already processed: {done_count} / {len(images)}\n")

    for img_path in images:
        key = img_path.name

        if results.get(key, {}).get("status") == "done":
            print(f"  ⏭️  {key}  (already done)")
            skipped += 1
            continue

        print(f"  🔍  {key} …", end=" ", flush=True)

        try:
            b64, mime  = image_to_base64(img_path)
            metadata   = extract_metadata(img_path)

            # Step 1: English description
            en = call_openrouter([{
                "role": "user",
                "content": [
                    {"type": "image_url",
                     "image_url": {"url": f"data:{mime};base64,{b64}"}},
                    {"type": "text",
                     "text": (
                         "Describe this image in detail. "
                         "Cover the main subject, setting, colors, mood, and any notable details. "
                         "Write in plain continuous prose paragraphs only. "
                         "Do not use markdown, headers, bullet points, bold, or any formatting."
                     )}
                ]
            }])

            # Step 2: Lithuanian translation
            lt = call_openrouter([{
                "role": "user",
                "content": (
                    "Translate the following text to Lithuanian. "
                    "Output only the translated text in plain prose. "
                    "Do not add any preamble, explanation, or formatting:\n\n"
                    + en
                )
            }])

            results[key] = {
                "status":         "done",
                "file":           str(img_path),
                "metadata":       metadata,
                "description_en": en,
                "description_lt": lt,
            }
            print(f"✅  ({len(metadata)} metadata fields)")
            processed += 1

        except Exception as e:
            results[key] = {"status": "error", "file": str(img_path), "error": str(e)}
            print(f"❌  {e}")
            failed += 1

        save_results(results)

    print(f"\n✨  Done!  processed={processed}  skipped={skipped}  failed={failed}")
    print(f"📄  Output: {OUTPUT_JSON}")
    return results

results = process_images()

Mounted at /content/drive
📁  Drive base : /content/drive/MyDrive/team
🖼️   Photos dir : /content/drive/MyDrive/team/photos
📄  Output JSON: /content/drive/MyDrive/team/descriptions.json

📂  Found 150 image(s) in 'photos'
📋  Already processed: 150 / 150

  ⏭️  20251110_202541.jpg  (already done)
  ⏭️  20260101_181421.jpg  (already done)
  ⏭️  20260515_092646.jpg  (already done)
  ⏭️  20260515_092655.jpg  (already done)
  ⏭️  20260515_092833.jpg  (already done)
  ⏭️  20260515_092836.jpg  (already done)
  ⏭️  20260515_093244.jpg  (already done)
  ⏭️  20260515_094057.jpg  (already done)
  ⏭️  20260515_094424.jpg  (already done)
  ⏭️  20260515_094827.jpg  (already done)
  ⏭️  20260515_095011.jpg  (already done)
  ⏭️  20260515_095040.jpg  (already done)
  ⏭️  20260515_095826.jpg  (already done)
  ⏭️  20260515_103207.jpg  (already done)
  ⏭️  20260515_103239.jpg  (already done)
  ⏭️  20260515_104644.jpg  (already done)
  ⏭️  20260515_110526.jpg  (already done)
  ⏭️  20260515_111108.jpg  (alrea